In [ ]:
import pandas as pd
import numpy as np
import folium
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print("⚙️ FASE 1: Preparando Datos Maestros...")

# 1. Limpieza y Cruce
def limpiar_socio(path, pivot_col):
    df = pd.read_csv(path)
    if 'Poblacion' in path:
        df.columns = [str(col).split()[-1] if '01 ene' in str(col).lower() else col for col in df.columns]
        df = df[df['Tipo de territorio'].isin(['Municipi', 'Districte'])]
    
    cols_num = [c for c in df.columns if c not in ['Territorio', 'Tipo de territorio', pivot_col]]
    for col in cols_num:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False).str.replace('-', '0', regex=False).replace('', '0').astype(float)
    
    melted = df.melt(id_vars=['Territorio', 'Tipo de territorio', pivot_col], var_name='Año', value_name='Val')
    melted['Año'] = pd.to_numeric(melted['Año'], errors='coerce')
    return melted.pivot_table(index=['Territorio', 'Año'], columns=pivot_col, values='Val', aggfunc='first').reset_index()

df_edad = limpiar_socio('Edad_Barrios.csv', 'Edad en grandes grupos')
df_pob = limpiar_socio('Poblacion-Inmigrante.csv', 'Nacionalidad (España/UE/Resto extranjero)')
df_crime = pd.read_csv('Criminalidad_Mensual_Estructurada.csv')

df_m = pd.merge(df_crime, df_edad, on=['Territorio', 'Año'], how='left')
df_m = pd.merge(df_m, df_pob, on=['Territorio', 'Año'], how='left')

# Adaptar nombres para XGBoost
df_m = df_m.rename(columns={'<16 años': 'Menos_16_anios', '≥65 años': 'Mas_65_anios', '16-64 años': 'Entre_16_64_anios'})
social_cols = ['Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

df_m = df_m.sort_values(['Territorio', 'Categoría_Delito', 'Año', 'Mes_Num'])
df_m[social_cols] = df_m.groupby(['Territorio', 'Categoría_Delito'])[social_cols].ffill()
df_m = df_m.dropna(subset=social_cols).reset_index(drop=True)

# Lags y Medias de Tendencia
for i in [1, 2, 3, 12]:
    df_m[f'Lag_{i}'] = df_m.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].shift(i)

df_m['Media_3_meses'] = df_m.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].rolling(3).mean().shift(1).reset_index(level=[0,1], drop=True)
df_m['Media_6_meses'] = df_m.groupby(['Territorio', 'Categoría_Delito'])['Cantidad'].rolling(6).mean().shift(1).reset_index(level=[0,1], drop=True)

# ==========================================
# FASE 2: MOTOR XGBOOST DE ÉLITE (Toda la ciudad)
# ==========================================
print("🚀 FASE 2: Proyectando 2026 con Inteligencia XGBoost (Error ~6.6%)...")

features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_12', 'Media_3_meses', 'Media_6_meses', 'Mes_Num', 
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

resultados_totales = []
for distrito in df_m['Territorio'].unique():
    for delito in df_m['Categoría_Delito'].unique():
        data_train = df_m[(df_m['Territorio'] == distrito) & (df_m['Categoría_Delito'] == delito)].dropna().copy()
        
        if len(data_train) > 24:
            # 🏆 LA CONFIGURACIÓN GANADORA INYECTADA DIRECTAMENTE
            xgb_opt = XGBRegressor(
                n_estimators=300,
                max_depth=2,
                learning_rate=0.01,
                subsample=0.6,
                colsample_bytree=1.0,
                reg_lambda=1,
                random_state=42,
                n_jobs=-1
            )
            xgb_opt.fit(data_train[features], data_train['Cantidad'])
            
            historial = df_m[(df_m['Territorio'] == distrito) & (df_m['Categoría_Delito'] == delito) & (df_m['Año'] <= 2025)].sort_values(['Año', 'Mes_Num'])['Cantidad'].tolist()
            ultimo_ctx = data_train.iloc[-1]
            
            for mes in range(1, 13):
                m3, m6 = np.mean(historial[-3:]), np.mean(historial[-6:])
                
                feats = np.array([[historial[-1], historial[-2], historial[-3], historial[-12], 
                                   m3, m6, mes, ultimo_ctx['Entre_16_64_anios'], ultimo_ctx['Menos_16_anios'], 
                                   ultimo_ctx['Mas_65_anios'], ultimo_ctx['España'], ultimo_ctx['Resto del mundo']]])
                
                pred = int(max(0, xgb_opt.predict(feats)[0]))
                historial.append(pred)
                
                resultados_totales.append({'Territorio': distrito, 'Categoría_Delito': delito, 'Año': 2026, 'Mes_Num': mes, 'Cantidad_Proyectada': pred})

df_2026_master = pd.DataFrame(resultados_totales)
df_2026_master.to_excel('Informe_Definitivo_XGBoost_2026.xlsx', index=False)
print("✅ Excel Maestro guardado con éxito.")

# ==========================================
# FASE 3: MAPA INTERACTIVO PROFESIONAL
# ==========================================
print("🗺️ FASE 3: Generando Mapa de Riesgo Interactivo...")

df_total = df_2026_master.groupby('Territorio')['Cantidad_Proyectada'].sum().reset_index()
top_delitos = df_2026_master.groupby(['Territorio', 'Categoría_Delito'])['Cantidad_Proyectada'].sum().reset_index()
top_delitos = top_delitos.sort_values(['Territorio', 'Cantidad_Proyectada'], ascending=[True, False])

coordenadas = {
    'Ciutat Vella': [41.3828, 2.1769], 'Eixample': [41.3889, 2.1611],
    'Sants-Montjuïc': [41.3630, 2.1431], 'Les Corts': [41.3861, 2.1264],
    'Sarrià-Sant Gervasi': [41.4011, 2.1128], 'Gràcia': [41.4096, 2.1575],
    'Horta-Guinardó': [41.4283, 2.1561], 'Nou Barris': [41.4416, 2.1772],
    'Sant Andreu': [41.4358, 2.1897], 'Sant Martí': [41.4055, 2.1975]
}

mapa_bcn = folium.Map(location=[41.3950, 2.1600], zoom_start=12, tiles='cartodbpositron')

p33, p66 = np.percentile(df_total['Cantidad_Proyectada'], 33), np.percentile(df_total['Cantidad_Proyectada'], 66)

for index, row in df_total.iterrows():
    distrito = row['Territorio']
    total = int(row['Cantidad_Proyectada'])
    
    if distrito in coordenadas:
        lat, lon = coordenadas[distrito]
        
        if total > p66: color, f_color, riesgo = '#e74c3c', '#c0392b', 'ALTO'
        elif total > p33: color, f_color, riesgo = '#f39c12', '#d35400', 'MEDIO'
        else: color, f_color, riesgo = '#2ecc71', '#27ae60', 'BAJO'
            
        radio = np.sqrt(total) / 2.5 
        
        delitos_zona = top_delitos[top_delitos['Territorio'] == distrito].head(4)
        html_popup = f"""
        <div style="font-family: Arial, sans-serif; width: 260px;">
            <h3 style="margin-bottom: 5px; color: {f_color}; text-align: center;">{distrito.upper()}</h3>
            <div style="background-color: {f_color}; color: white; text-align: center; padding: 3px; border-radius: 4px; font-weight: bold; font-size: 11px;">
                NIVEL DE RIESGO: {riesgo}
            </div>
            <p style="margin-top: 10px; font-size: 13px; text-align: center;">
                <b>Total Previsto 2026:</b> <span style="color:#333; font-size: 16px;">{total}</span>
            </p>
            <hr style="border-top: 1px solid #eee;">
            <b style="font-size: 12px; color: #555;">🚨 Top 4 Delitos:</b>
            <table style="width: 100%; font-size: 11px; margin-top: 8px; border-collapse: collapse;">
        """
        for _, d_row in delitos_zona.iterrows():
            nombre = d_row['Categoría_Delito'][:22] + ".." if len(d_row['Categoría_Delito']) > 22 else d_row['Categoría_Delito']
            html_popup += f"<tr style='border-bottom: 1px solid #f1f1f1;'><td style='padding: 4px 0;'>{nombre}</td><td style='text-align: right; font-weight: bold; color: #333;'>{int(d_row['Cantidad_Proyectada'])}</td></tr>"
            
        html_popup += "</table></div>"
        
        iframe = folium.IFrame(html=html_popup, width=290, height=210)
        popup_interactivo = folium.Popup(iframe, max_width=290)
        
        folium.CircleMarker(
            location=[lat, lon],
            radius=radio,
            color=color,
            fill=True,
            fill_color=f_color,
            fill_opacity=0.7,
            weight=2,
            popup=popup_interactivo,
            tooltip=f"Ver predicción para {distrito}"
        ).add_to(mapa_bcn)

archivo = 'Mapa_XGBoost_Optimizado_Barcelona_2026.html'
mapa_bcn.save(archivo)
print(f"🌟 ¡SISTEMA FINALIZADO! Abre '{archivo}' para visualizar el panel geoespacial predictivo.")

mapa_bcn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

print("⚙️ Entrenando LightGBM OPTIMIZADO (El algoritmo de Microsoft)...")

DISTRITO = 'Eixample' 
DELITO = 'Hurto'

# 1. Preparar datos (Asumimos que df_m está en memoria con sus Lags y Medias)
data_eval = df_m[(df_m['Territorio'] == DISTRITO) & (df_m['Categoría_Delito'] == DELITO)].dropna().copy()

features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_12', 'Media_3_meses', 'Media_6_meses', 'Mes_Num', 
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

X = data_eval[features].values
y = data_eval['Cantidad'].values

# Dividimos en 80% Entrenamiento y 20% Test (Igual que con XGBoost para que sea justo)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 2. Definir el "Freno y Acelerador" para LightGBM
param_dist_lgb = {
    'num_leaves': [7, 15, 31],               # CLAVE en LightGBM: pocas hojas evitan memorizar ruido
    'max_depth': [-1, 3, 5],                 # -1 significa sin límite (se controla con num_leaves)
    'learning_rate': [0.01, 0.05, 0.1],      # Ritmo de aprendizaje
    'n_estimators': [100, 200, 300],         # Cantidad de árboles
    'subsample': [0.6, 0.8, 1.0],            # Muestreo de filas
    'colsample_bytree': [0.6, 0.8, 1.0],     # Muestreo de columnas
    'reg_lambda': [1, 5, 10]                 # Penalización matemática
}

# 3. Búsqueda Automática con Validación Temporal
tscv = TimeSeriesSplit(n_splits=3)

# Instanciamos LightGBM (verbose=-1 apaga los mensajes internos de la librería)
lgbm_base = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

buscador_lgb = RandomizedSearchCV(
    estimator=lgbm_base, 
    param_distributions=param_dist_lgb, 
    n_iter=30,               # Probará 30 configuraciones distintas
    cv=tscv, 
    scoring='neg_mean_absolute_percentage_error',
    random_state=42,
    n_jobs=-1
)

print("🔎 LightGBM buscando su estructura óptima de hojas...")
buscador_lgb.fit(X_train, y_train)

mejor_lgb = buscador_lgb.best_estimator_

print(f"🏆 LightGBM ha elegido esta configuración:")
print(buscador_lgb.best_params_)

# 4. Evaluación del Nuevo Motor LightGBM
y_pred_lgb = mejor_lgb.predict(X_test)
y_pred_lgb = [max(0, p) for p in y_pred_lgb] # Evitar absurdos negativos

mape_lgb = mean_absolute_percentage_error(y_test, y_pred_lgb) * 100

print(f"\n📉 RESULTADO LIGHTGBM:")
print(f"Error Porcentual (MAPE): {mape_lgb:.2f}%")

# 5. Gráfica de Precisión (Real vs XGBoost vs LightGBM)
# Suponiendo que aún tienes 'y_pred_xgb_opt' del bloque anterior en memoria. 
# Si no lo tienes, la gráfica solo mostrará LightGBM y la realidad.
plt.figure(figsize=(12, 5))
plt.plot(y_test, label='Realidad Histórica', color='navy', marker='o', linewidth=2)
plt.plot(y_pred_lgb, label=f'LightGBM ({mape_lgb:.2f}%)', color='magenta', linestyle='--', marker='s', linewidth=2)

# Intentamos pintar el XGBoost si existe en tu sesión
try:
    plt.plot(y_pred_xgb_opt, label='XGBoost Optimizado (6.62%)', color='darkorange', linestyle=':', marker='x', linewidth=2)
except NameError:
    pass

plt.title(f"El Duelo Final: LightGBM vs Realidad ({DELITO} en {DISTRITO})", fontsize=14)
plt.ylabel("Incidentes")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Seleccionamos las columnas que queremos analizar
columnas_analisis = [
    'Cantidad', 'Entre_16_64_anios', 'Menos_16_anios', 
    'Mas_65_anios', 'España', 'Resto del mundo'
]

# 2. Calculamos la correlación de Pearson
# (Mide la relación lineal entre -1 y 1)
corr_matrix = df_m[columnas_analisis].corr()

# 3. Visualización con Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt=".2f", linewidths=0.5)

plt.title("Matriz de Correlación: Factores Sociales vs. Criminalidad", fontsize=15, pad=20)
plt.show()

# 4. Análisis rápido por consola
print("📊 IMPACTO DIRECTO SOBRE LA CANTIDAD DE DELITOS:")
impacto = corr_matrix['Cantidad'].sort_values(ascending=False).drop('Cantidad')
for var, val in impacto.items():
    estado = "Relación Positiva 📈" if val > 0 else "Relación Negativa 📉"
    print(f" - {var}: {val:.2f} ({estado})")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("📊 GENERANDO RANKING DE RIESGO: PROYECCIONES 2026")

# 1. Recuperamos las predicciones generadas en el paso anterior
# Si por algún motivo se borró de la memoria, lo lee del Excel que acabamos de crear:
try:
    df_plot = df_2026.copy()
except NameError:
    df_plot = pd.read_excel('Predicciones_Hurtos_Barcelona_2026.xlsx')

# 2. Agrupamos los datos: Sumamos todos los meses para tener el total anual por distrito
ranking_2026 = df_plot.groupby('Territorio')['Predicción_Incidentes'].sum().reset_index()

# 3. Ordenamos de mayor a menor número de incidentes para el gráfico
ranking_2026 = ranking_2026.sort_values(by='Predicción_Incidentes', ascending=True)

# 4. Configuración visual del gráfico (Estilo profesional)
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

# Usamos una paleta de colores tipo "Heat" (Rojo para el de mayor riesgo, bajando la intensidad)
colores = sns.color_palette("Reds", n_colors=len(ranking_2026))

# Crear el gráfico de barras horizontales (facilita la lectura de los nombres)
barras = plt.barh(ranking_2026['Territorio'], ranking_2026['Predicción_Incidentes'], color=colores)

# 5. Añadir las etiquetas de datos (los números exactos al final de cada barra)
for barra in barras:
    ancho = barra.get_width()
    plt.text(ancho + (ancho * 0.01),  # Posición X (un poco a la derecha del final de la barra)
             barra.get_y() + barra.get_height()/2,  # Posición Y (centro de la barra)
             f'{int(ancho):,}',  # Texto (número con separador de miles)
             va='center', ha='left', fontsize=11, fontweight='bold', color='black')

# 6. Títulos y ajustes finales
plt.title("Proyección Anual de Hurtos por Distrito (Barcelona, 2026)", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Volumen Total de Incidentes Previstos", fontsize=12, labelpad=10)
plt.ylabel("Distrito", fontsize=12)

# Eliminar el marco superior y derecho para un diseño más limpio
sns.despine()

# Ajustar los márgenes para que no se corte ningún texto
plt.tight_layout()

# Guardar la imagen en alta calidad para tu informe
nombre_imagen = 'Ranking_Hurtos_2026.png'
plt.savefig(nombre_imagen, dpi=300, bbox_inches='tight')
print(f"✅ Gráfico guardado exitosamente como '{nombre_imagen}'")

# Mostrar el gráfico en pantalla
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
import logging

# Ocultar mensajes molestos de Prophet
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

print("🥊 ¡COMIENZA EL DUELO FINAL: Prophet vs CatBoost!")

DISTRITO = 'Eixample' 
DELITO = 'Hurto'

# 1. Preparar datos base (Asumimos df_m en memoria)
data_eval = df_m[(df_m['Territorio'] == DISTRITO) & (df_m['Categoría_Delito'] == DELITO)].dropna().copy()

# ==========================================
# RETADOR 1: CATBOOST (Machine Learning Tabular)
# ==========================================
print("\n⚙️ Entrenando CatBoost...")
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_12', 'Media_3_meses', 'Media_6_meses', 'Mes_Num', 
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

X = data_eval[features].values
y = data_eval['Cantidad'].values

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# CatBoost es tan potente que lo probaremos con su configuración "de fábrica" (con un toque conservador)
cat = CatBoostRegressor(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    loss_function='RMSE',
    verbose=0, # Silenciar el entrenamiento
    random_seed=42
)

cat.fit(X_train, y_train)
y_pred_cat = cat.predict(X_test)
y_pred_cat = [max(0, p) for p in y_pred_cat]

mape_cat = mean_absolute_percentage_error(y_test, y_pred_cat) * 100
print(f"✅ CatBoost Terminado. MAPE: {mape_cat:.2f}%")

# ==========================================
# RETADOR 2: PROPHET (Estadística de Meta)
# ==========================================
print("\n⚙️ Entrenando Prophet (Preparando fechas reales)...")

# Prophet obliga a tener una columna de fecha exacta llamada 'ds' y el objetivo llamado 'y'
data_eval['Fecha'] = pd.to_datetime(data_eval['Año'].astype(str) + '-' + data_eval['Mes_Num'].astype(str) + '-01')
df_prophet = data_eval[['Fecha', 'Cantidad', 'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']].copy()
df_prophet = df_prophet.rename(columns={'Fecha': 'ds', 'Cantidad': 'y'})

train_prophet = df_prophet.iloc[:split]
test_prophet = df_prophet.iloc[split:]

# Inicializamos Prophet
m_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)

# Le añadimos nuestras variables sociales como "Regresores Extra"
regresores = ['Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']
for reg in regresores:
    m_prophet.add_regressor(reg)

m_prophet.fit(train_prophet)

# Predecir
pred_prophet = m_prophet.predict(test_prophet.drop(columns='y'))
y_pred_pro = pred_prophet['yhat'].values
y_pred_pro = [max(0, p) for p in y_pred_pro]

mape_pro = mean_absolute_percentage_error(test_prophet['y'].values, y_pred_pro) * 100
print(f"✅ Prophet Terminado. MAPE: {mape_pro:.2f}%")

# ==========================================
# RESULTADO FINAL Y GRÁFICA
# ==========================================
print("\n🏆 --- MARCADOR FINAL ---")
print(f"LightGBM (Campeón Actual) : 5.71%")
print(f"CatBoost                  : {mape_cat:.2f}%")
print(f"Prophet                   : {mape_pro:.2f}%")

plt.figure(figsize=(12, 5))
plt.plot(y_test, label='Realidad Histórica', color='navy', marker='o', linewidth=2)
plt.plot(y_pred_cat, label=f'CatBoost ({mape_cat:.2f}%)', color='red', linestyle='--', marker='^', linewidth=2)
plt.plot(y_pred_pro, label=f'Prophet ({mape_pro:.2f}%)', color='green', linestyle=':', marker='s', linewidth=2)

# Referencia del campeón
plt.axhline(np.mean(y_test), color='gray', linestyle='-', alpha=0.3)

plt.title(f"Evaluación de Modelos: {DELITO} en {DISTRITO}", fontsize=14)
plt.ylabel("Incidentes")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

print("🧠 CONSTRUYENDO EL METAMODELO DE ÉLITE (Stacking)...")

DISTRITO = 'Eixample' 
DELITO = 'Hurto'

# 1. Rescatar datos
data_eval = df_m[(df_m['Territorio'] == DISTRITO) & (df_m['Categoría_Delito'] == DELITO)].dropna().copy()
features = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_12', 'Media_3_meses', 'Media_6_meses', 'Mes_Num', 
            'Entre_16_64_anios', 'Menos_16_anios', 'Mas_65_anios', 'España', 'Resto del mundo']

X = data_eval[features].values
y = data_eval['Cantidad'].values

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 2. Definir los Modelos Base 
lgb = LGBMRegressor(num_leaves=15, learning_rate=0.05, n_estimators=200, random_state=42, verbose=-1)
xgb = XGBRegressor(max_depth=3, learning_rate=0.05, n_estimators=200, random_state=42, n_jobs=-1)
cat = CatBoostRegressor(depth=4, learning_rate=0.05, iterations=200, verbose=0, random_seed=42)

modelos_base = [
    ('LightGBM', lgb),
    ('XGBoost', xgb),
    ('CatBoost', cat)
]

# 3. Definir el Metamodelo
metamodelo = Ridge(alpha=1.0)

# 4. Construir y Entrenar la Arquitectura Stacking
# SOLUCIÓN: Usamos cv=5 interno (KFold normal) para satisfacer la matemática del Stacking
stacking_regressor = StackingRegressor(
    estimators=modelos_base,
    final_estimator=metamodelo,
    cv=5, 
    n_jobs=-1
)

print("⏳ Entrenando a los 3 algoritmos y al Metamodelo... (Puede tardar unos segundos)")
stacking_regressor.fit(X_train, y_train)

# 5. Predicción y Evaluación
y_pred_stack = stacking_regressor.predict(X_test)
y_pred_stack = [max(0, p) for p in y_pred_stack]

mape_stack = mean_absolute_percentage_error(y_test, y_pred_stack) * 100

print(f"\n🏆 RESULTADO DEL METAMODELO (STACKING):")
print(f"Error Porcentual (MAPE): {mape_stack:.2f}%")

# Extraer los pesos para ver en quién confía más el Jefe
pesos = stacking_regressor.final_estimator_.coef_
print("\n⚖️ ¿A quién le hace más caso el Metamodelo?")
print(f" - Peso otorgado a LightGBM: {pesos[0]:.4f}")
print(f" - Peso otorgado a XGBoost:  {pesos[1]:.4f}")
print(f" - Peso otorgado a CatBoost: {pesos[2]:.4f}")

# 6. Gráfica de Precisión
plt.figure(figsize=(11, 5))
plt.plot(y_test, label='Realidad Histórica', color='navy', marker='o')
plt.plot(y_pred_stack, label=f'Metamodelo Stacking ({mape_stack:.2f}%)', color='purple', linestyle='-', marker='D', linewidth=2, alpha=0.8)
plt.title(f"La Predicción Definitiva: {DELITO} en {DISTRITO}", fontsize=14, fontweight='bold')
plt.ylabel("Número de Incidentes")
plt.legend()
plt.grid(alpha=0.3)
plt.show()